# Federation Demo  
In this demo, you will be shown:  
- how to import data from endpoints
- how to see the datasets imported
- how to search for data
- merge data from different tables
- add a table to an existing database

In [1]:
import csv
import os
from getpass import getpass
from pathlib import Path
from typing import Tuple, List, Dict, Any

import getpass
import paramiko
import shutil
import json
import pandas as pd
from pathlib import Path

from dsi.dsi import DSI
from dsi.dsifederated import DSIFederated
from dsi.sync import Sync
from dsi.utils.federated.federate_datasets import (
    pull_data, 
    just_pull_data, 
    pull_remote_db, 
    get_remote_endpoints,
    pull_data_endpoints
)

from dsi.utils.federation_utils import (
    compute_md5, 
    create_directory, 
    create_hashed_folder_from_path, 
    create_folder,
    combine_csv,
    csv_to_list_of_dicts, 
    deduplicate_keep_latest, 
    get_last_part, 
    human_readable_size, 
    should_download, 
    upsert_records
)

Help for DSI and Federated DSI

In [2]:
help(DSI)

Help on class DSI in module dsi.dsi:

class DSI(builtins.object)
 |  DSI(filename='.temp_dsi.db', backend_name='Sqlite', **kwargs)
 |
 |  A user-facing interface for DSI's Core middleware.
 |
 |  The DSI Class abstracts Core.Terminal for managing metadata and Core.Sync for data management and movement.
 |
 |  Methods defined here:
 |
 |  __init__(self, filename='.temp_dsi.db', backend_name='Sqlite', **kwargs)
 |      Initializes DSI by activating a backend for data operations; default is a Sqlite backend for temporary data analysis.
 |      If users specify `filename`, data is saved to a permanent backend file.
 |
 |      `filename` : str, optional, default is ".temp_dsi.db"
 |          If not specified, a temporary, hidden backend file is created for users to analyze their data.
 |          If specified and backend file already exists, it is activated for a user to explore its data.
 |          If specified and backend file does not exist, a file with this name is created.
 |
 |      

In [3]:
help(DSIFederated)

Help on class DSIFederated in module dsi.dsifederated:

class DSIFederated(builtins.object)
 |  DSIFederated(federated_folder_path: str, operating_mode: str = 'console')
 |
 |  A class for federated querying of DSI databases. It loads metadata about the databases and
 |  their tables from a specified folder, and provides methods to summarize, query, search, and find data across the federated databases.
 |
 |  Methods defined here:
 |
 |  __init__(self, federated_folder_path: str, operating_mode: str = 'console')
 |      Initializes the DSIFederated class by loading metadata about the federated databases and their tables from a specified folder.
 |
 |      Args:
 |          federated_folder_path (str): The file path to the folder containing the metadata about the federated databases. The folder should contain a JSON file named "dsi_database_list.json" with the metadata information.
 |          operating_mode (str): console or notebook, determines how the results are displayed. Default i

In [4]:
help(Sync)

Help on class Sync in module dsi.sync:

class Sync(builtins.object)
 |  Sync(project_name, isVerbose=False, no_parent=False, skip_index=False, **kwargs)
 |
 |  A class defined to assist in data management activities for DSI
 |
 |  Sync is where data movement functions such as copy (to remote location) and
 |  sync (local filesystem with remote) exist.
 |
 |  Methods defined here:
 |
 |  __init__(self, project_name, isVerbose=False, no_parent=False, skip_index=False, **kwargs)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  change_group(self, local_loc, user_group)
 |      Change group permissions for data and db. Only works for OS with Unix (not Windows)
 |
 |  change_permissions(self, local_loc)
 |      Change read permissions for data and db. Only works for OS with Unix (not Windows)
 |
 |  copy(self, tool='copy')
 |      Helper function to perform the data copy over using a preferred API
 |
 |  dircrawl(self, filepath, verbose=False)
 |      Crawls the 

## Get Enpoints from HPC

In [5]:
hpc_name = input("Enter the name of the HPC")
username = input("Enter username: ")
password = getpass.getpass("Enter password: ")

# currently a script setting environment variables but should be load module in the future
script_path='/users/pascalgrosset/dsi_test/load_dsi_endpoints.sh' 

# prefix of the endpoints; environment variables to search for
prefixes=['DSI_ENDPOINT_', 'DIANA_ENDPOINT_'] 

endpoints_location = get_remote_endpoints(hpc_name, username, password, script_path, prefixes)

Enter the name of the HPC ch-fe.lanl.gov
Enter username:  pascalgrosset
Enter password:  ········


Connecting to ch-fe.lanl.gov...
Sourcing /users/pascalgrosset/dsi_test/load_dsi_endpoints.sh and reading endpoints...
✓ Found 2 endpoints:
  DSI_ENDPOINT_CHICOMA_2 = /users/pascalgrosset/dsi_test/dsi_online_sources.csv
  DSI_ENDPOINT_CHICOMA_1 = /users/pascalgrosset/dsi_test/dsi_hpc_sources.csv


In [6]:
endpoints_location

{'DSI_ENDPOINT_CHICOMA_2': '/users/pascalgrosset/dsi_test/dsi_online_sources.csv',
 'DSI_ENDPOINT_CHICOMA_1': '/users/pascalgrosset/dsi_test/dsi_hpc_sources.csv'}

### Federate the data in specified folder

In [ ]:
rel_wrks_folder = "test_federate_07"
workspace_folder = str(Path(rel_wrks_folder).resolve())

database_info = pull_data_endpoints(endpoints_location, hpc_name, workspace_folder)

In [ ]:
database_info

## Instantiate the object

In [ ]:
federated_dbs = DSIFederated(workspace_folder, operating_mode="notebook")

## Browse and search through the data

In [ ]:
federated_dbs.f_list_databases()

## Looking up data

In [ ]:
federated_dbs.f_summary()

In [ ]:
federated_dbs.f_search(query="dens_max")

## Merging data

### Search for databases

In [ ]:
federated_dbs.f_search_for_databases(db="data_subset*")

### Search for the path to a database

In [ ]:
federated_dbs.f_get_db_path(db="data_subset_1.db")

### Load that database

In [ ]:
temp_file = DSI('/Users/pascalgrosset/projects/dsi/dsi_databases_00/87c1361f3c2316dd/data_subset_1.db')

In [ ]:
temp_file.list()

In [ ]:
temp_file.get_table("data", collection=True)

In [ ]:
federated_dbs.f_merge(src_db_id='camouflaged-hare',src_tbl_name='data', dst_db_id='aromatic-dragon',dst_tbl_name='data')

In [ ]:
temp_file.get_table("data", collection=True)

In [ ]:
federated_dbs.f_merge(src_db_id='precious-walrus',src_tbl_name='data', dst_db_id='aromatic-dragon',dst_tbl_name='data')

In [ ]:
temp_file.get_table("data", collection=True)

## Adding another table to the database

In [ ]:
federated_dbs.f_search_for_databases(db="model_subset*")

In [ ]:
federated_dbs.f_add_table(src_db_id='satisfied-whale',src_tbl_name='data', dst_db_id='aromatic-dragon',dst_tbl_name='model')

In [ ]:
temp_file.list()

In [ ]:
temp_file.get_table("model", collection=True)

That database now has datasets which have been pulled from different sites as well as several tables 